# Post-OFAT ML All Models + Tuning: OLLAMA

Ноутбук читает готовые OFAT feature-артефакты, оценивает модели и запускает тюнинг для лучших схем.


In [ ]:
import json
import math
import os
import re
import time
import warnings
from pathlib import Path

_mpl_cache_dir = Path(os.getenv("MPLCONFIGDIR", "/tmp/matplotlib-cache"))
_mpl_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(_mpl_cache_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    BaggingRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import GridSearchCV, ParameterGrid, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("xgboost is required. xgboost is required") from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


In [ ]:
SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"
PROVIDER = "ollama"
RUN_LABEL = "post_ofat_all_ml_models_with_tuning"

# Set these to True if you intentionally want to overwrite existing outputs.
FORCE_RERUN_BASELINE = False
FORCE_RERUN_TUNING = False

def require_path(env_name: str, label: str) -> Path:
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        raise FileNotFoundError(f"Укажи путь к {label} через переменную окружения {env_name}.")
    path = Path(raw_value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


def optional_path(env_name: str, default_value: str | Path) -> Path:
    return Path(os.environ.get(env_name, default_value)).expanduser()

PROJECT_DIR = optional_path("OLLAMA_OFAT_PROJECT_DIR", "./ollama_meta_prompt_ofat_fixed_features")
INPUT_DIR = optional_path("LLM_INPUT_DIR", PROJECT_DIR / "inputs")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
MODEL_TAG = re.sub(r"[^a-zA-Z0-9._-]+", "_", OLLAMA_MODEL).replace(":", "_")
DATA_PATH = require_path("DATA_FINALL_PATH", "data_finall")
ARTIFACT_ROOT = optional_path("OLLAMA_OFAT_ARTIFACT_ROOT", "./ollama_meta_prompt_ofat_fixed_features/artifacts_fixed_features_v7_llama3.2_3b")
OUTPUT_DIR = optional_path("POST_OFAT_ML_OLLAMA_OUTPUT_DIR", "./artifacts_post_ofat_ml_all_models_with_tuning_ollama")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [DATA_PATH]
ARTIFACT_ROOT_CANDIDATES = [ARTIFACT_ROOT]

print("Provider:", PROVIDER)
print("Output dir:", OUTPUT_DIR.resolve())


In [ ]:
def resolve_existing_path(candidates, label):
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"{label}: {p.resolve()}")
            return p
    raise FileNotFoundError(f"{label} not found. Tried: {[str(Path(p)) for p in candidates]}")


DATA_PATH = resolve_existing_path(DATA_CANDIDATES, "Data")
ARTIFACT_ROOT = resolve_existing_path(ARTIFACT_ROOT_CANDIDATES, "OFAT artifact root")


df = pd.read_csv(DATA_PATH).copy()
df["source_row_id"] = np.arange(len(df), dtype=int)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy().reset_index(drop=True)
train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
val_start = int(len(train_filt) * 0.75)
train_core = train_filt.iloc[:val_start].copy().reset_index(drop=True)
val_fresh = train_filt.iloc[val_start:].copy().reset_index(drop=True)
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

meta_train_core = train_core.drop(columns=[TARGET_COL], errors="ignore").copy()
meta_val_fresh = val_fresh.drop(columns=[TARGET_COL], errors="ignore").copy()
meta_test_full = test_full.drop(columns=[TARGET_COL], errors="ignore").copy()

y_train_core = train_core[TARGET_COL].reset_index(drop=True)
y_val_fresh = val_fresh[TARGET_COL].reset_index(drop=True)
y_train_all = pd.concat([y_train_core, y_val_fresh], ignore_index=True)
y_test_full = test_full[TARGET_COL].reset_index(drop=True)
y_test_typical = test_typical[TARGET_COL].reset_index(drop=True)

print("full df      :", df.shape)
print("train_filt   :", train_filt.shape)
print("train_core   :", train_core.shape)
print("val_fresh    :", val_fresh.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)


In [ ]:
FEATURE_COLS = [
    "execution_complexity_1_5",
    "coordination_risk_1_5",
    "testing_effort_1_5",
    "uncertainty_1_5",
    "urgent_delivery_0_1",
    "likely_long_task_0_1",
]

STAGE_ORDER = [
    ("stage_1_role", 1),
    ("stage_2_project", 2),
    ("stage_3_goal", 3),
    ("stage_4_granularity", 4),
    ("stage_5_long_task_policy", 5),
]


def reg_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MdAE": median_absolute_error(y_true, pred),
    }


def read_agg(config_dir, split_name):
    path = Path(config_dir) / f"aggregated_features__{split_name}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path)


def prepare_model_matrix(meta_df, agg_df):
    merged = meta_df.merge(agg_df, on="source_row_id", how="left").copy()
    for c in FEATURE_COLS:
        if c not in merged.columns:
            merged[c] = 0.0
    drop_cols = [
        c
        for c in merged.columns
        if c.endswith("__std")
        or c.endswith("__agree")
        or c.endswith("__p1")
        or c.endswith("__var")
        or c in {"raw_text", "latency_s", "mean_latency_s"}
    ]
    merged = merged.drop(columns=drop_cols, errors="ignore")
    merged = merged.drop(columns=[TARGET_COL], errors="ignore")
    merged = merged.select_dtypes(include=[np.number, "bool"]).copy()
    merged = merged.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return merged


def discover_ofat_configs(artifact_root):
    rows = []
    for stage_name, stage_num in STAGE_ORDER:
        stage_dir = Path(artifact_root) / stage_name
        if not stage_dir.exists():
            print("Missing stage dir:", stage_dir)
            continue
        for config_dir in sorted(p for p in stage_dir.iterdir() if p.is_dir()):
            required = [
                config_dir / "aggregated_features__train_core.csv",
                config_dir / "aggregated_features__val_fresh.csv",
                config_dir / "aggregated_features__test_full.csv",
            ]
            if all(p.exists() for p in required):
                rows.append({
                    "provider": PROVIDER,
                    "stage": stage_name,
                    "stage_num": stage_num,
                    "config_slug": config_dir.name,
                    "config_dir": config_dir,
                })
    out = pd.DataFrame(rows).sort_values(["stage_num", "config_slug"]).reset_index(drop=True)
    if len(out) != 20:
        print(f"WARNING: expected 20 OFAT configs, found {len(out)}")
    return out


def load_config_matrices(config_dir):
    agg_train_core = read_agg(config_dir, "train_core")
    agg_val_fresh = read_agg(config_dir, "val_fresh")
    agg_test_full = read_agg(config_dir, "test_full")
    if agg_train_core is None or agg_val_fresh is None or agg_test_full is None:
        raise FileNotFoundError(f"Config is missing required aggregate files: {config_dir}")

    X_train_core = prepare_model_matrix(meta_train_core, agg_train_core)
    X_val_fresh = prepare_model_matrix(meta_val_fresh, agg_val_fresh)
    X_train_all = pd.concat([X_train_core, X_val_fresh], ignore_index=True)

    X_test_full = prepare_model_matrix(meta_test_full, agg_test_full)
    typical_ids = set(test_typical["source_row_id"].tolist())
    X_test_typical = X_test_full.loc[test_full["source_row_id"].isin(typical_ids)].reset_index(drop=True)
    return X_train_all, X_test_typical, X_test_full


def walk_forward_cv_mae(estimator, X, y, n_splits=3):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_maes = []
    for tr_idx, va_idx in tscv.split(X):
        est = clone(estimator)
        est.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = est.predict(X.iloc[va_idx])
        fold_maes.append(mean_absolute_error(y.iloc[va_idx], pred))
    return np.array(fold_maes, dtype=float)


def normalize_model_name(model_name):
    return str(model_name).replace("Scaled_", "")


def extract_core_estimator(estimator):
    return estimator.steps[-1][1] if isinstance(estimator, Pipeline) else estimator


def extract_model_params(estimator):
    core = extract_core_estimator(estimator)
    clean = {}
    for k, v in core.get_params(deep=False).items():
        clean[k] = v.item() if hasattr(v, "item") else v
    return clean


ofat_configs = discover_ofat_configs(ARTIFACT_ROOT)
ofat_configs.to_csv(OUTPUT_DIR / "ofat_configs_discovered.csv", index=False)
display(ofat_configs[["stage", "stage_num", "config_slug", "config_dir"]])


## Baseline models from ML_baseline

In [ ]:
def make_baseline_pipelines(seed=SEED):
    """Exact 15-model list from ML_baseline.ipynb."""
    pipelines = []
    pipelines.append(("Scaled_Lasso", Pipeline([("Scaler", StandardScaler()), ("Lasso", Lasso(random_state=seed))])))
    pipelines.append(("Scaled_Elastic", Pipeline([("Scaler", StandardScaler()), ("Elastic", ElasticNet(random_state=seed))])))
    pipelines.append(("Scaled_RF_reg", Pipeline([("Scaler", StandardScaler()), ("RF", RandomForestRegressor(random_state=seed))])))
    pipelines.append(("Scaled_ET_reg", Pipeline([("Scaler", StandardScaler()), ("ET", ExtraTreesRegressor(random_state=seed))])))
    pipelines.append(("Scaled_BR_reg", Pipeline([("Scaler", StandardScaler()), ("BR", BaggingRegressor(random_state=seed))])))
    pipelines.append(("Scaled_DT_reg", Pipeline([("Scaler", StandardScaler()), ("DT_reg", DecisionTreeRegressor(random_state=seed))])))
    pipelines.append(("Scaled_Ridge", Pipeline([("Scaler", StandardScaler()), ("Ridge", Ridge(random_state=seed))])))
    pipelines.append(("Scaled_SVR", Pipeline([("Scaler", StandardScaler()), ("SVR", SVR(kernel="linear", C=1e2))])))
    pipelines.append(("Scaled_Hub-Reg", Pipeline([("Scaler", StandardScaler()), ("Hub-Reg", HuberRegressor())])))
    pipelines.append(("Scaled_BayRidge", Pipeline([("Scaler", StandardScaler()), ("BR", BayesianRidge())])))
    pipelines.append(("Scaled_KNN_reg", Pipeline([("Scaler", StandardScaler()), ("KNN_reg", KNeighborsRegressor())])))
    pipelines.append(("Scaled_Gboost-Reg", Pipeline([("Scaler", StandardScaler()), ("GBoost-Reg", GradientBoostingRegressor(random_state=seed))])))
    pipelines.append(("Scaled_XGB_reg", Pipeline([("Scaler", StandardScaler()), ("XGBR", XGBRegressor(random_state=seed))])))
    pipelines.append(("Scaled_RFR_PCA", Pipeline([("Scaler", StandardScaler()), ("PCA", PCA(n_components=3)), ("RF", RandomForestRegressor(random_state=seed))])))
    pipelines.append(("Scaled_XGBR_PCA", Pipeline([("Scaler", StandardScaler()), ("PCA", PCA(n_components=3)), ("XGB", XGBRegressor(random_state=seed))])))
    return pipelines


pipelines = make_baseline_pipelines()
print("Baseline models from ML_baseline:", len(pipelines))
display(pd.DataFrame({"model": [name for name, _ in pipelines], "model_norm": [normalize_model_name(name) for name, _ in pipelines]}))


## Run all ML_baseline models on all 20 OFAT configs

In [ ]:
BASELINE_PATH = OUTPUT_DIR / "all_20_ofat_baseline_models.csv"
BEST_SCHEME_PATH = OUTPUT_DIR / "best_ofat_scheme_by_model.csv"
BEST_BY_CONFIG_PATH = OUTPUT_DIR / "best_model_by_ofat_config.csv"

if BASELINE_PATH.exists() and not FORCE_RERUN_BASELINE:
    print("Loading existing baseline results:", BASELINE_PATH.resolve())
    baseline_results = pd.read_csv(BASELINE_PATH)
else:
    baseline_rows = []
    expected_runs = len(ofat_configs) * len(pipelines)
    run_i = 0

    for cfg_i, cfg in ofat_configs.iterrows():
        print(f"\n[{cfg_i + 1}/{len(ofat_configs)}] {cfg['stage']} :: {cfg['config_slug']}")
        X_train_all, X_test_typical, X_test_full = load_config_matrices(cfg["config_dir"])
        print("  X_train_all:", X_train_all.shape, "X_test_typical:", X_test_typical.shape, "X_test_full:", X_test_full.shape)

        for model_name, estimator in pipelines:
            run_i += 1
            t0 = time.time()
            fold_maes = walk_forward_cv_mae(estimator, X_train_all, y_train_all)
            fitted = clone(estimator)
            fitted.fit(X_train_all, y_train_all)
            pred_typ = fitted.predict(X_test_typical)
            pred_full = fitted.predict(X_test_full)
            typ = reg_metrics(y_test_typical, pred_typ)
            full = reg_metrics(y_test_full, pred_full)
            row = {
                "provider": PROVIDER,
                "stage": cfg["stage"],
                "stage_num": int(cfg["stage_num"]),
                "config_slug": cfg["config_slug"],
                "config_dir": str(cfg["config_dir"]),
                "model": model_name,
                "model_norm": normalize_model_name(model_name),
                "CV_MAE_mean": float(fold_maes.mean()),
                "CV_MAE_std": float(fold_maes.std()),
                "CV_MAE_folds": json.dumps([round(float(x), 6) for x in fold_maes]),
                "holdout_typical_MAE": float(typ["MAE"]),
                "holdout_typical_RMSE": float(typ["RMSE"]),
                "holdout_typical_MdAE": float(typ["MdAE"]),
                "holdout_full_MAE": float(full["MAE"]),
                "holdout_full_RMSE": float(full["RMSE"]),
                "holdout_full_MdAE": float(full["MdAE"]),
                "fit_seconds": round(time.time() - t0, 3),
            }
            baseline_rows.append(row)
            print(f"  [{run_i:03d}/{expected_runs}] {model_name:<20s} CV MAE={row['CV_MAE_mean']:.2f} +/- {row['CV_MAE_std']:.2f}")

        pd.DataFrame(baseline_rows).to_csv(OUTPUT_DIR / "all_20_ofat_baseline_models_progress.csv", index=False)

    baseline_results = (
        pd.DataFrame(baseline_rows)
        .sort_values(["CV_MAE_mean", "CV_MAE_std", "holdout_typical_MAE"])
        .reset_index(drop=True)
    )
    baseline_results.to_csv(BASELINE_PATH, index=False)

best_scheme_by_model = (
    baseline_results
    .sort_values(["model_norm", "CV_MAE_mean", "CV_MAE_std", "holdout_typical_MAE"])
    .groupby("model_norm", as_index=False)
    .head(1)
    .sort_values(["CV_MAE_mean", "CV_MAE_std", "holdout_typical_MAE"])
    .reset_index(drop=True)
)
best_scheme_by_model.to_csv(BEST_SCHEME_PATH, index=False)

best_by_config = (
    baseline_results
    .sort_values(["stage_num", "config_slug", "CV_MAE_mean", "CV_MAE_std"])
    .groupby(["stage", "stage_num", "config_slug"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_by_config.to_csv(BEST_BY_CONFIG_PATH, index=False)

print("Baseline rows:", len(baseline_results))
print("Best scheme by model saved to:", BEST_SCHEME_PATH.resolve())
display(best_scheme_by_model[["model", "model_norm", "stage", "config_slug", "CV_MAE_mean", "CV_MAE_std", "holdout_typical_MAE", "holdout_full_MAE"]])


## Tune each model on its best OFAT scheme with ML_tuning grids

In [ ]:
def make_tuning_estimators(seed=SEED):
    """Seven tuned model families and grids copied from ML_tuning.ipynb."""
    return {
        "Lasso": Pipeline([("Scaler", StandardScaler()), ("Lasso", Lasso(random_state=seed))]),
        "Elastic": Pipeline([("Scaler", StandardScaler()), ("Elastic", ElasticNet(random_state=seed))]),
        "Ridge": Pipeline([("Scaler", StandardScaler()), ("Ridge", Ridge(random_state=seed))]),
        "BayRidge": Pipeline([("Scaler", StandardScaler()), ("BR", BayesianRidge())]),
        "Hub-Reg": Pipeline([("Scaler", StandardScaler()), ("Hub-Reg", HuberRegressor())]),
        "Gboost-Reg": GradientBoostingRegressor(random_state=seed),
        "XGB_reg": XGBRegressor(objective="reg:absoluteerror", eval_metric="mae", random_state=seed, tree_method="hist", n_jobs=-1),
    }


TUNING_PARAM_GRIDS = {
    "Lasso": {
        "Lasso__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 16), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]]), 6).tolist())),
        "Lasso__selection": ["cyclic", "random"],
        "Lasso__max_iter": [100000],
    },
    "Elastic": {
        "Elastic__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 14), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]]), 6).tolist())),
        "Elastic__l1_ratio": [0.70, 0.85, 0.90, 0.95, 0.98, 0.995],
        "Elastic__max_iter": [100000],
    },
    "Ridge": {
        "Ridge__alpha": np.logspace(-2, 3, 20).tolist(),
    },
    "BayRidge": {
        "BR__alpha_1": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__alpha_2": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__lambda_1": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__lambda_2": [1e-8, 1e-6, 1e-4, 1e-2],
    },
    "Hub-Reg": {
        "Hub-Reg__epsilon": [1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5],
        "Hub-Reg__alpha": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
        "Hub-Reg__max_iter": [10000],
    },
    "Gboost-Reg": [
        {
            "loss": ["squared_error", "absolute_error"],
            "n_estimators": [100, 150],
            "learning_rate": [0.05, 0.10],
            "max_depth": [2, 3, 4],
            "min_samples_leaf": [1, 4],
            "min_samples_split": [2, 8],
            "subsample": [0.70, 1.0],
            "max_features": [None, 1.0],
        },
        {
            "loss": ["huber"],
            "alpha": [0.90],
            "n_estimators": [100, 150],
            "learning_rate": [0.05, 0.10],
            "max_depth": [2, 3, 4],
            "min_samples_leaf": [1, 4],
            "min_samples_split": [2, 8],
            "subsample": [0.70, 1.0],
            "max_features": [None, 1.0],
        },
    ],
    "XGB_reg": {
        "n_estimators": [300, 600],
        "learning_rate": [0.02, 0.05],
        "max_depth": [2, 3],
        "min_child_weight": [5, 10],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_alpha": [0.0, 0.1],
        "reg_lambda": [1.0, 5.0],
        "gamma": [0.0, 1.0],
        "objective": ["reg:absoluteerror"],
        "eval_metric": ["mae"],
        "tree_method": ["hist"],
        "random_state": [SEED],
        "n_jobs": [-1],
    },
}


tuning_estimators = make_tuning_estimators()
models_without_ml_tuning_grid = sorted(set(best_scheme_by_model["model_norm"]) - set(TUNING_PARAM_GRIDS))
print("Models with ML_tuning grid:", sorted(TUNING_PARAM_GRIDS))
print("Baseline-only models without ML_tuning grid:", models_without_ml_tuning_grid)


In [ ]:
TUNING_PATH = OUTPUT_DIR / "best_scheme_grid_tuning_results.csv"
TUNING_SKIP_PATH = OUTPUT_DIR / "baseline_models_without_ml_tuning_grid.csv"

tuning_skip = best_scheme_by_model[best_scheme_by_model["model_norm"].isin(models_without_ml_tuning_grid)].copy()
tuning_skip.to_csv(TUNING_SKIP_PATH, index=False)

if TUNING_PATH.exists() and not FORCE_RERUN_TUNING:
    print("Loading existing tuning results:", TUNING_PATH.resolve())
    tuning_results = pd.read_csv(TUNING_PATH)
else:
    tscv = TimeSeriesSplit(n_splits=3)
    tuning_rows = []

    for _, best_row in best_scheme_by_model.sort_values("CV_MAE_mean").iterrows():
        model_norm = best_row["model_norm"]
        if model_norm not in TUNING_PARAM_GRIDS:
            print(f"\nSkipping {model_norm}: no grid in ML_tuning.ipynb")
            continue

        print(f"\n===== Tuning {model_norm} on best OFAT scheme: {best_row['stage']} :: {best_row['config_slug']} =====")
        config_dir = Path(best_row["config_dir"])
        X_train_all, X_test_typical, X_test_full = load_config_matrices(config_dir)
        estimator = tuning_estimators[model_norm]
        param_grid = TUNING_PARAM_GRIDS[model_norm]
        n_combos = len(list(ParameterGrid(param_grid)))
        print(f"Grid: {n_combos} combinations x 3 folds")

        t0 = time.time()
        grid = GridSearchCV(
            estimator=clone(estimator),
            param_grid=param_grid,
            scoring="neg_mean_absolute_error",
            cv=tscv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        grid.fit(X_train_all, y_train_all)
        elapsed = time.time() - t0

        pred_typ = grid.best_estimator_.predict(X_test_typical)
        pred_full = grid.best_estimator_.predict(X_test_full)
        typ = reg_metrics(y_test_typical, pred_typ)
        full = reg_metrics(y_test_full, pred_full)
        cv_mae = -grid.best_score_

        cv_dump = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]].copy()
        cv_dump["mean_test_MAE"] = -cv_dump["mean_test_score"]
        cv_dump = cv_dump.sort_values("rank_test_score")
        cv_dump.to_csv(OUTPUT_DIR / f"grid_cv_best_scheme_{model_norm}.csv", index=False)

        row = {
            "provider": PROVIDER,
            "model_norm": model_norm,
            "source_baseline_model": best_row["model"],
            "stage": best_row["stage"],
            "stage_num": int(best_row["stage_num"]),
            "config_slug": best_row["config_slug"],
            "config_dir": str(config_dir),
            "baseline_best_scheme_CV_MAE_mean": float(best_row["CV_MAE_mean"]),
            "baseline_best_scheme_CV_MAE_std": float(best_row["CV_MAE_std"]),
            "baseline_best_scheme_holdout_typical_MAE": float(best_row["holdout_typical_MAE"]),
            "baseline_best_scheme_holdout_full_MAE": float(best_row["holdout_full_MAE"]),
            "grid_cv_MAE": float(cv_mae),
            "grid_holdout_typical_MAE": float(typ["MAE"]),
            "grid_holdout_typical_RMSE": float(typ["RMSE"]),
            "grid_holdout_typical_MdAE": float(typ["MdAE"]),
            "grid_holdout_full_MAE": float(full["MAE"]),
            "grid_holdout_full_RMSE": float(full["RMSE"]),
            "grid_holdout_full_MdAE": float(full["MdAE"]),
            "delta_cv_vs_baseline_best_scheme": float(best_row["CV_MAE_mean"] - cv_mae),
            "delta_typical_vs_baseline_best_scheme": float(best_row["holdout_typical_MAE"] - typ["MAE"]),
            "fit_seconds": round(elapsed, 3),
            "best_params": json.dumps(extract_model_params(grid.best_estimator_), ensure_ascii=False, default=str),
        }
        tuning_rows.append(row)
        pd.DataFrame(tuning_rows).to_csv(OUTPUT_DIR / "best_scheme_grid_tuning_results_progress.csv", index=False)
        print(f"CV MAE: {cv_mae:.2f}; typical MAE: {typ['MAE']:.2f}; full MAE: {full['MAE']:.2f}; time: {elapsed:.1f}s")

    tuning_results = (
        pd.DataFrame(tuning_rows)
        .sort_values(["grid_cv_MAE", "grid_holdout_typical_MAE"])
        .reset_index(drop=True)
    )
    tuning_results.to_csv(TUNING_PATH, index=False)

print("Tuned rows:", len(tuning_results))
display(tuning_results)


## Final report

In [ ]:
top30 = baseline_results.head(30).copy()
top30["label"] = top30["model"] + " | " + top30["stage"].str.replace("stage_", "s", regex=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=top30, x="CV_MAE_mean", y="label", hue="stage", dodge=False)
plt.title(f"{PROVIDER}: top baseline candidates across 20 OFAT configs")
plt.xlabel("Walk-forward CV MAE")
plt.ylabel("")
plt.tight_layout()
plt.show()

if len(tuning_results):
    plt.figure(figsize=(10, 5))
    tune_plot = tuning_results.copy()
    tune_plot["label"] = tune_plot["model_norm"] + " | " + tune_plot["stage"].str.replace("stage_", "s", regex=False)
    sns.barplot(data=tune_plot, x="grid_cv_MAE", y="label", color="#4c78a8")
    plt.title(f"{PROVIDER}: grid tuning on each model's best OFAT scheme")
    plt.xlabel("GridSearchCV MAE")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

report = {
    "provider": PROVIDER,
    "data_path": str(DATA_PATH),
    "artifact_root": str(ARTIFACT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "n_ofat_configs": int(len(ofat_configs)),
    "n_baseline_models": int(len(pipelines)),
    "n_expected_baseline_runs": int(len(ofat_configs) * len(pipelines)),
    "n_baseline_rows": int(len(baseline_results)),
    "n_models_with_best_scheme": int(len(best_scheme_by_model)),
    "n_tuned_models": int(len(tuning_results)),
    "models_without_ml_tuning_grid": models_without_ml_tuning_grid,
    "best_baseline_overall": baseline_results.head(1).to_dict(orient="records"),
    "best_tuned_overall": tuning_results.head(1).to_dict(orient="records") if len(tuning_results) else [],
}

with open(OUTPUT_DIR / "run_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print(json.dumps(report, ensure_ascii=False, indent=2, default=str))
